# Query understanding - Oracle

An Oracle helps us understand how good we could do if our classification was perfect. We essentially cheat then see how good NDCG could be.

## Boilerplate

Install deps, mount GDrive, prompt for your OpenAI Key (placed in your GDrive)

In [1]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git
!pip install typesafe-sdk
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)
from cheat_at_search.search import run_strategy, graded_bm25, ndcgs, ndcg_delta, vs_ideal
from cheat_at_search.wands_data import corpus, judgments

corpus

  Cloning https://github.com/softwaredoug/cheat-at-search.git to /tmp/pip-req-build-u3hu9xjb
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /tmp/pip-req-build-u3hu9xjb
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit 611269f6b5f0857681d774e990312c512f0e9ca2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 22.2 MB/s eta 0:00:00
  Created wheel for cheat_at_search: filename=cheat_at_search-0.1.0-py3-none-any.whl size=1455859 sha256=9fbc458cc6597cd26cd75ac76b488c6253a8e802d67264f16710d2ca36fa6855
  Stored in directory: /tmp/pip-ephem-wheel-cache-k186_r8n/wheels/ce/b7/b5/5b6456ba5de93e3b9f0b94ddbeb32ac5a07345c

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,doc_id,title,description,category,sub_category,cat_subcat,image_url,title_snowball,description_snowball
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0,"[overallwidth-sidetoside:64.7, dsprimaryproduc...",0,solid wood platform bed,"good , deep sleep can be quite difficult to ha...",Furniture,Bedroom Furniture,Furniture / Bedroom Furniture,https://storage.googleapis.com/product-ai-imag...,"Terms({'solid', 'bed', 'wood', 'platform'})","Terms({'there', 'quit', 'for', 'one', 'well', ..."
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0,"[capacityquarts:7, producttype : slow cooker, ...",1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend...",Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances,https://storage.googleapis.com/product-ai-imag...,"Terms({'all', 'slow', 'qt', '7', 'clad', 'cook...","Terms({'soup', 'electr', 'one', 'aluminum', 'd..."
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0,"[features : keep warm setting, capacityquarts:...",2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...,Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances,https://storage.googleapis.com/product-ai-imag...,"Terms({'all', 'slow', '5', 'qt', 'cooker', 'cl...","Terms({'ani', 'slow', 'cook', 'insert', 'timer..."
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0,"[overallwidth-sidetoside:3.5, warrantylength :...",3,all-clad all professional tools pizza cutter,this original stainless tool was designed to c...,Browse By Brand,All-Clad,Browse By Brand / All-Clad,https://storage.googleapis.com/product-ai-imag...,"Terms({'all', 'pizza', 'profession', 'clad', '...","Terms({'pizza', 's', 'design', 'complement', '..."
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0,"[compatibledoorthickness:1.375 '' , countryofo...",4,baldwin prestige alcott passage knob with roun...,the hardware has a rich heritage of delivering...,Home Improvement,Doors & Door Hardware,Home Improvement / Doors & Door Hardware,https://storage.googleapis.com/product-ai-imag...,"Terms({'prestig', 'alcott', 'round', 'passag',...","Terms({'confid', 'deliv', 'homeown', 'inspir',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42989,42989,malibu pressure balanced diverter fixed shower...,Shower Panels,Home Improvement / Bathroom Remodel & Bathroom...,the malibu pressure balanced diverter fixed sh...,producttype : shower panel|spraypattern : rain...,3.0,4.5,2.0,"[producttype : shower panel, spraypattern : ra...",42989,malibu pressure balanced diverter fixed shower...,the malibu pressure balanced diverter fixed sh...,Home Improvement,Bathroom Remodel & Bathroom Fixtures,Home Improvement / Bathroom Remodel & Bathroom...,https://storage.googleapis.com/product-ai-imag...,"Terms({'divert', 'fix', 'shower', 'panel', 'ma...","Terms({'singl', 'ani', 'shower', 'hand', 'oasi..."
42990,42990,emmeline 5 piece breakfast dining set,Dining Table Sets,Furniture / Kitchen 

## Build an oracle classifier

In [7]:
def oracle(query, rel_grade=2):
    query_judgments = judgments[judgments['query'] == query]
    max_grade = query_judgments['grade'].max()
    min_grade = query_judgments['grade'].min()
    if max_grade == min_grade:
        return []
    relevant_docs = query_judgments[query_judgments['grade'] == max_grade]
    relevant_doc_ids = relevant_docs['doc_id'].tolist()
    relevant_docs = corpus[corpus['doc_id'].isin(relevant_doc_ids)]

    classifications = []
    for candidate in relevant_docs['category hierarchy'].unique().tolist():
        if isinstance(candidate, str) and candidate.strip() != '':
            classifications.append(candidate)
    return classifications

oracle('salon chair')

['Commercial Business Furniture / Commercial Office Furniture / Office Seating / Waiting Room Chairs',
 'Furniture / Living Room Furniture / Chairs & Seating / Recliners',
 'Furniture / Living Room Furniture / Chairs & Seating / Massage Chairs',
 'Commercial Business Furniture / Commercial Office Furniture / Office Seating / Waiting Room Chairs / Metal Waiting Room Chairs',
 'Furniture / Office Furniture / Office Chairs',
 'Commercial Business Furniture / Commercial Office Furniture / Office Seating / Waiting Room Chairs / Wood Waiting Room Chairs',
 'Shop Product Type / Chairs / Guest & Reception Chairs']

## Run Category search strategy with classifier

Let's search, providing a boost when the product matches the category

In [9]:
from searcharray import SearchArray
from cheat_at_search.wands_data import corpus, judgments
from cheat_at_search.tokenizers import snowball_tokenizer
from cheat_at_search.strategy.strategy import SearchStrategy
import numpy as np


def parse_category(classification: str) -> str | None:
    toked = classification.split('/')
    if len(toked) > 1:
        return toked[0]
    return None

def parse_sub_category(classification: str) -> str | None:
    toked = classification.split('/')
    if len(toked) > 1:
        return '/'.join(toked[1:])
    return None

class CategorySearch(SearchStrategy):
    def __init__(self, products, query_to_cat,
                 name_boost=9.3,
                 description_boost=4.1,
                 category_boost=0,
                 sub_category_boost=0,
                 direct_boost=10):
        super().__init__(products, workers=16)
        self.index = products
        if 'title_snowball' not in self.index.columns:
            self.index['title_snowball'] = SearchArray.index(
                products['title'], snowball_tokenizer)
        if 'description_snowball' not in self.index.columns:
            self.index['description_snowball'] = SearchArray.index(
                products['description'], snowball_tokenizer)

        cat_split = products['category hierarchy'].fillna('').str.split("/")
        self.index['category_snowball'] = SearchArray.index(
            products['category'], snowball_tokenizer
        )
        self.index['sub_category_snowball'] = SearchArray.index(
            products['sub_category'], snowball_tokenizer
        )
        self.index['classification_snowball'] = SearchArray.index(
            products['category hierarchy'], snowball_tokenizer
        )

        self.query_to_cat = query_to_cat
        self.name_boost = name_boost
        self.description_boost = description_boost
        self.category_boost = category_boost
        self.sub_category_boost = sub_category_boost
        self.direct_boost = direct_boost

    def search(self, query, k=10):
        """Dumb baseline lexical search, but add a constant boost when
           the desired category or subcategory"""
        bm25_scores = np.zeros(len(self.index))
        classifications = self.query_to_cat(query)
        categories = [parse_category(classification) for classification in classifications]
        sub_categories = [parse_sub_category(classification) for classification in classifications]

        categories = set(categories)
        sub_categories = set(sub_categories)

        tokenized = snowball_tokenizer(query)

        # ****
        # Baseline BM25 search from before
        for token in tokenized:
            bm25_scores += self.index['title_snowball'].array.score(token) * self.name_boost
            bm25_scores += self.index['description_snowball'].array.score(
                token) * self.description_boost

        # ****
        # If there's a category, boost that by a constant amount
        for category in categories:
            tokenized_category = snowball_tokenizer(category)
            category_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_category:
                category_match = self.index['category_snowball'].array.score(tokenized_category) > 0
            bm25_scores[category_match] += self.category_boost

        for sub_category in sub_categories:
            tokenized_sub_category = snowball_tokenizer(sub_category)
            sub_category_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_sub_category:
                sub_category_match = self.index['category_snowball'].array.score(tokenized_sub_category) > 0
            bm25_scores[sub_category_match] += self.sub_category_boost

        # ***
        for classification in classifications:
            tokenized_classification = snowball_tokenizer(classification)
            classification_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_classification:
                classification_match = self.index['classification_snowball'].array.score(tokenized_classification) > 0
            bm25_scores[classification_match] += self.direct_boost

        top_k = np.argsort(-bm25_scores)[:k]
        scores = bm25_scores[top_k]

        return top_k, scores

oracle_search = CategorySearch(corpus, oracle)
graded_oracle = run_strategy(oracle_search, judgments)
graded_oracle

2026-10-07 02:30:21,267 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-10-07 02:30:21,275 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-10-07 02:30:21,279 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

2026-10-07 02:30:25,678 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-10-07 02:30:25,709 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete
Searching: 100%|██████████| 480/480 [00:43<00:00, 10.92it/s]


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,...,score,query,query_id,rank,grade,discounted_gain,idcg,dcg,ndcg,mrr
0,5750,14 '' hood duct vent,Large Appliance Accessories,Appliances / Kitchen Appliances / Large Applia...,the broan vent hood duct transition provides a...,colorfinish : galvanized steel|countryoforigin...,20.0,4.5,20.0,"[colorfinish : galvanized steel, countryoforig...",...,104.843268,farmhouse hood vent,381,1,1.0,1.000000,8.786905,2.436111,0.277243,0.0
1,9599,range hood vent,Large Appliance Accessories,Appliances / Kitchen Appliances / Large Applia...,keep out unwanted birds with the universal bir...,overallproductweight:0.15|colorfinish : white|...,NaN,NaN,NaN,"[overallproductweight:0.15, colorfinish : whit...",...,93.543396,farmhouse hood vent,381,2,1.0,0.500000,8.786905,2.436111,0.277243,0.0
2,5749,range hood 10 '' round vent,Large Appliance Accessories,Appliances / Kitchen Appliances / Large Applia...,broan 's round transition can maximize the air...,overallheight-toptobottom:10.13|overalldepth-f...,40.0,5.0,40.0,"[overallheight-toptobottom:10.13, overalldepth...",...,82.182343,farmhouse hood vent,381,3,1.0,0.333333,8.786905,2.436111,0.277243,0.0
3,30807,greenhouse auto vent kit,Greenhouse Accessories,Outdoor / Garden / Greenhouse Supplies,the automatic vent opener was designed to auto...,overallwidth-sidetoside:2.7|toolsneededforinst...,111.0,4.5,83.0,"[overallwidth-sidetoside:2.7, toolsneededforin...",...,59.752750,farmhouse hood vent,381,4,0.0,0.000000,8.786905,2.436111,0.277243,0.0
4,42490,48 '' hand crafted designer wood island hood s...,Range Hoods,Appliances / Kitchen Appliances / Range Hoods ...,the zline is a 48 in . range hood with a remot...,maximumsones:5.3|hoodheight:13.25|overallheigh...,175.0,5.0,175.0,"[maximumsones:5.3, hoodheight:13.25, overallhe...",...,54.412974,farmhouse hood vent,381,5,1.0,0.200000,8.786905,2.436111,0.277243,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,6389,rinehart mid century modern dining chair,Dining Chairs,Sale / Closeout / Kitchen & Dining Chairs / Ar...,be amazed by the comfort and style this rineha...,weightcapacity:225|style : modern & contempora...,2.0,4.5,2.0,"[weightcapacity:225, style : modern & contempo...",...,101.310694,mid century modern,149,6,2.0,0.500000,8.786905,8.786905,1.000000,1.0
4796,14692,markson mid-century modern server,Sideboards & Buffets,Furniture / Kitchen & Dining Furniture / Sideb...,"from your kitchen to your foyer , our sideboar...",levelofassembly : partial assembly|overallheig...,30.0,4.5,25.0,"[levelofassembly : partial assembly, overallhe...",...,101.288912,mid century modern,149,7,2.0,0.428571,8.786905,8.786905,1.000000,1.0
4797,26445,chauvin mid century modern nightstand,Nightstands,Furniture / Bedroom Furniture / Nightstands,a stylish sidekick in both your living room or...,shelfweightcapacity:11|dsprimaryproductstyle :...,3.0,2.5,3.0,"[shelfweightcapacity:11, dsprimaryproductstyle...",...,100.577589,mid century modern,149,8,2.0,0.375000,8.786905,8.786905,1.000000,1.0
4798,8403,chyna mid century modern arm chair in white,Dining Chairs,Furniture / Kitchen & Dining Furniture / Dinin...,"this dining armchair will be a rich , contempo...",overallwidth-sidetoside:24.5|overallproductwei...,3.0,5.0,3.0,"[overallwidth-sidetoside:24.5, overallproductw...",...,98.524085,mid century modern,149,9,2.0,0.333333,8.786905,8.786905,1.000000,1.0


### Analyze results

* We notice a small bump in performance in NDCG from GPT-5.
* We also notice a bigger bump with Jev

In [10]:
ndcgs(graded_bm25).mean(), ndcgs(graded_oracle).mean()

(np.float64(0.5411098691836396), np.float64(0.5825066988815125))